# Full example → одна траектория → BF16 master
Учитель **Qwen3-4B**, student **Qwen3-1.7B**, исходный промпт Hogwild с `<think>`.
Периодические вставки `insert_s1_prompt_every_tokens` отключены.
Для этой задачи выбран бюджет **450 раундов**: оба агента успевают получить ответ,
а последующий повторяющийся хвост не попадает в обучение. Это не универсальный детектор завершения.
Здесь сохранён общий барьер после каждого токена; отдельный эксперимент — в `no_barrier.ipynb`.
Основной код генерации и обучения — в ячейках; служебные функции — в `async_helpers.py`.
Kernel **AsyncLLM (uv)** → **Run All**. Настройки GPU рассчитаны на RTX A4000 в imladris.


In [1]:
import gc
import os
import re
import sys
import time
from contextlib import aclosing
from pathlib import Path
from tempfile import TemporaryDirectory

from IPython.display import Markdown, display

# Start Jupyter in the repository root or scripts/parallel_thinking.
ROOT = Path.cwd().parents[1] if Path.cwd().name == "parallel_thinking" else Path.cwd()
os.chdir(ROOT)
for path in (ROOT / "python", ROOT / "scripts/async_thoughts", ROOT / "scripts/parallel_thinking"):
    sys.path.insert(0, str(path))
os.environ.update(
    CUDA_DEVICE_ORDER="PCI_BUS_ID",
    CUDA_VISIBLE_DEVICES="2",
    TVM_FFI_CUDA_ARCH_LIST="8.6",
    OMP_NUM_THREADS="8",
)
os.environ["PATH"] = f"{Path(sys.executable).parent}:/usr/local/cuda/bin:" + os.environ["PATH"]

import torch
from async_helpers import isolated, one_token, recorded_example, together
from async_thoughts.training import DemoRecorder
from hogwild_formatting import MathFormatting
from minisgl.core import SamplingParams
from minisgl.llm import AsyncLLM
from minisgl.shared_cache import AsyncContext
from minisgl.training.replay import Replay
from minisgl.training.sft import evaluate, supervised_loss
from transformers import AutoModelForCausalLM, AutoTokenizer

TEACHER = "models/Qwen3-4B"
STUDENT = "models/Qwen3-1.7B"
WORKERS = ("Alice", "Bob")
PROBLEM = (
    "Three vertices of a cube are $P=(7,12,10)$ , $Q=(8,8,1)$ , and "
    "$R=(11,3,9)$ . What is the surface area of the cube?"
)

/home/uskovev/async_llm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Учитель
Alice читает `[common, header, Bob, separator, Alice]`, Bob — обратный порядок.
Завершённый шаг переносится в общий блок. Recorder хранит реальные batch, видимость и мутации кеша.
У обычной Qwen3-4B шаблон сам не добавляет `<think>`: явно восстанавливаем префикс
успешного контрольного промпта. Веса и общий текст задачи не меняются.
Учитель запускается в отдельном процессе, чтобы освободить GPU перед обучением.


In [2]:
async def solve_parallel(model, *, max_steps=450, prompt_ids=None):
    started = time.perf_counter()
    llm = AsyncLLM(
        str(model),
        dtype=torch.bfloat16,
        memory_ratio=0.65,
        max_running_req=4,
        max_seq_len_override=16384,
        cuda_graph_bs=[1, 2],
        cuda_graph_max_bs=2,
        shared_cuda_graph_max_depth=6,
        shared_cuda_graph_prefill_rows=[32, 256],
        max_prefill_rows=256,
        distributed_addr="tcp://127.0.0.1:0",
    )
    llm = DemoRecorder(llm)
    blocks = []
    try:
        tokenizer = llm.tokenizer
        encode = lambda text: tokenizer.encode(text, add_special_tokens=False)
        # Match the original QwQ prompt, including the thinking marker and newlines.
        fmt = MathFormatting(
            tokenizer, WORKERS, generation_prefix="<think>\n" + MathFormatting.generation_prefix
        )
        for _ in range(6):
            blocks.append(await llm.create_block())
        common, others_header, own_header, alice, bob, answer_block = blocks
        worker_blocks = [alice, bob]
        contexts = [
            AsyncContext(cache_view=[common, others_header, bob, own_header, alice]),
            AsyncContext(cache_view=[common, others_header, alice, own_header, bob]),
        ]
        initial_ids = (
            list(prompt_ids) if prompt_ids is not None else encode(fmt.apply_chat_template(PROBLEM))
        )
        # Graph profiles admit two requests, so initialize the long prompt first.
        await llm.forward(initial_ids, write_to=common, return_logits=False)
        await together(
            llm.forward(
                encode(fmt.current_step_header), write_to=others_header, return_logits=False
            ),
            llm.forward(
                encode(fmt.current_worker_header), write_to=own_header, return_logits=False
            ),
        )

        prefixes = [
            f"{fmt.get_step_prefix(WORKERS[0], 1)}Hi, I'm {WORKERS[0]}. Here's how we can collaborate",
            f"{fmt.get_step_prefix(WORKERS[1], 1)}Hi, I'm {WORKERS[1]}.",
        ]
        current = [encode(prefix) for prefix in prefixes]
        # Leave the last prefix token pending, so round 1 is a joint decode.
        await together(
            *(
                llm.forward(
                    ids[:-1],
                    cache_view=[common, others_header, own_header, block],
                    return_logits=False,
                )
                for ids, block in zip(current, worker_blocks)
            )
        )
        for ctx, ids in zip(contexts, current):
            ctx.next_input_id = ids[-1]

        history, events = [], []
        step_indices, generated = [1, 1], [0, 0]
        sampled = [[], []]
        forbidden = [
            tokenizer.vocab[t]
            for t in ("<|im_start|>", "<|endoftext|>", "</think>")
            if t in tokenizer.vocab
        ]
        # EOS remains a stop condition, even if a model uses endoftext as EOS.
        eos = tokenizer.eos_token_id
        forbidden = [token for token in forbidden if token != eos]
        generation_started = time.perf_counter()

        for iteration in range(1, max_steps + 1):
            new_tokens = await together(*(one_token(llm, ctx, forbidden) for ctx in contexts))
            for i, token in enumerate(new_tokens):
                current[i].append(token)
                generated[i] += 1
                sampled[i].append(token)

            # Both futures are settled: mutations below cannot race a decode.
            finished = [i for i in range(2) if fmt.is_end_of_step(current[i])]
            await together(
                *(llm.forward(cache_view=contexts[i], return_logits=False) for i in finished)
            )
            for i in finished:
                block = worker_blocks[i]
                assert block.token_ids == current[i], "A boundary token was lost or duplicated."
                await llm.append_block(common, block)
                history.extend(current[i])
                assert common.token_ids == initial_ids + history
                events.append(
                    {
                        "round": iteration,
                        "worker": WORKERS[i],
                        "step": step_indices[i],
                        "tokens": len(current[i]),
                        "text": tokenizer.decode(current[i]),
                    }
                )
                await llm.free_block(block)  # Reset this same block; views keep their references.
                assert block.num_tokens == 0
                step_indices[i] += 1
                start = fmt.get_step_prefix(WORKERS[i], step_indices[i])
                current[i] = encode(start)

            # Prefill prefix[:-1] after all merges; keep the last token pending.
            await together(
                *(
                    llm.forward(current[i][:-1], cache_view=contexts[i], return_logits=False)
                    for i in finished
                )
            )
            for i in finished:
                contexts[i].next_input_id = current[i][-1]

            for i, ctx in enumerate(contexts):
                assert ctx.output_block.token_ids == current[i][:-1]
                assert ctx.next_input_id == current[i][-1]
            if iteration % 128 == 0:
                print(f"round {iteration}: {len(events)} completed steps", flush=True)
            if eos in new_tokens:
                break

        # Flush both final sampled tokens before the answer continuation reads them.
        await together(*(llm.forward(cache_view=ctx, return_logits=False) for ctx in contexts))
        for i, ctx in enumerate(contexts):
            assert ctx.output_block.token_ids == current[i]
        parallel_seconds = time.perf_counter() - generation_started
        transcript = fmt.history_header + fmt.sep + tokenizer.decode(history)
        transcript += fmt.sep + fmt.current_step_header
        transcript += "".join(
            tokenizer.decode(ids) + fmt.incomplete_step + fmt.sep for ids in current
        )

        suffix = fmt.s1_finisher_suffix
        answer_ids = encode(suffix)
        answer_ctx = AsyncContext(
            cache_view=[common, others_header, alice, own_header, bob, answer_block]
        )
        await llm.forward(answer_ids[:-1], cache_view=answer_ctx, return_logits=False)
        answer_ctx.next_input_id = answer_ids[-1]
        answer_tokens = []
        async with aclosing(
            llm.async_generate(
                answer_ctx,
                max_steps=64,
                forbid_ids=forbidden,
                sampling_params=SamplingParams(temperature=0.0, max_tokens=64),
            )
        ) as stream:
            async for token in stream:
                if token == eos:
                    break
                answer_tokens.append(token)
                if fmt.get_final_answer(suffix + tokenizer.decode(answer_tokens)) is not None:
                    break
        answer_text = suffix + tokenizer.decode(answer_tokens)
        result = {
            "model": str(model),
            "problem": PROBLEM,
            "rounds": iteration,
            "generated_tokens": dict(zip(WORKERS, generated)),
            "completed_steps": len(events),
            "merge_events": events,
            "transcript": transcript,
            "answer_text": answer_text,
            "answer": fmt.get_final_answer(answer_text),
            "parallel_seconds": parallel_seconds,
            "total_seconds": time.perf_counter() - started,
        }
        result["generated_text"] = {w: tokenizer.decode(ids) for w, ids in zip(WORKERS, sampled)}
        result["prompt_ids"] = initial_ids
        result["history_ids"] = history
        result["current_ids"] = current
        result["answer_ids"] = answer_tokens
        result["example"] = recorded_example(llm)
        return result
    finally:
        # No active streams or queued requests remain here, including on errors.
        try:
            for block in blocks:
                await llm.free_block(block)
        finally:
            await llm.close()
            gc.collect()
            torch.cuda.empty_cache()

In [3]:
teacher = isolated(solve_parallel, TEACHER)
example = teacher.pop("example")
display(Markdown(teacher["transcript"]))
print("Teacher answer:", teacher["answer"], "Published blocks:", teacher["completed_steps"])
role_headers = {
    worker: re.findall(r"\*\*(?:Alice|Bob) \[\d+\]:\*\*", text)
    for worker, text in teacher["generated_text"].items()
}
print("Model-generated role headers:", role_headers)
assert teacher["answer"] == 294
assert not any(role_headers.values()), "Inspect the teacher: a worker is writing dialogue headers."
# These checks do not measure cooperation: inspect the actual steps printed above.


### Past steps

**Bob [1]:** Hi, I'm Bob. I need to find the surface area of the cube given three of its vertices. Let me start by calculating the distances between the points P, Q, and R to see if they form edges, face diagonals, or space diagonals of the cube. 

**Alice [1]:** Hi, I'm Alice. Here's how we can collaborate. I'll start by calculating the distances between the given points to find the edge length of the cube. Let me calculate the distance between P and Q first. The distance formula in 3D is sqrt[(x2-x1)^2 + (y2-y1)^2 + (z2-z1)^2]. So for P=(7,12,10) and Q=(8,8,1), the distance is sqrt[(8-7)^2 + (8-12)^2 + (1-10)^2] = sqrt[1 + 16 + 81] = sqrt[98]. That's approximately 9.899. 

**Bob [2]:**  I'll calculate the distance between Q and R next. Then I can compare the distances to determine if they are edges, face diagonals, or space diagonals. Let me do that. The distance between Q=(8,8,1) and R=(11,3,9) is sqrt[(11-8)^2 + (3-8)^2 + (9-1)^2] = sqrt[9 + 25 + 64] = sqrt[98]. That's the same as the distance between P and Q. So both PQ and QR are sqrt(98). 

**Alice [2]:**  I see. Now I'll calculate the distance between P and R. P=(7,12,10) and R=(11,3,9). The distance is sqrt[(11-7)^2 + (3-12)^2 + (9-10)^2] = sqrt[16 + 81 + 1] = sqrt[98]. So all three distances are sqrt(98). That's interesting. So the three points are all the same distance apart. 

**Bob [3]:**  The distance between P and R is sqrt[(11-7)^2 + (3-12)^2 + (9-10)^2] = sqrt[16 + 81 + 1] = sqrt[98]. So all three distances are the same. That suggests that the three points form an equilateral triangle with side length sqrt(98). But in a cube, three vertices can form an equilateral triangle if they are connected by face diagonals or space diagonals. 

**Bob [4]:**  Let me think. If the three points are connected by edges, then the distance would be 'a'. If they are connected by face diagonals, the distance would be a√2. If they are connected by space diagonals, the distance would be a√3. Since sqrt(98) is approximately 9.899, and 7√2 is about 9.899, that suggests that the distance is a face diagonal. So the edge length of the cube would be 7. 

**Alice [3]:**  So if all three distances are equal, that means the three points are either all connected by edges, face diagonals, or space diagonals. But in a cube, the edges are of length 'a', face diagonals are a√2, and space diagonals are a√3. Since sqrt(98) is approximately 9.899, which is between a√2 and a√3 for a cube with a=7, since 7√2 ≈ 9.899 and 7√3 ≈ 12.124. So sqrt(98) is exactly 7√2. Therefore, the edge length of the cube is 7. 

**Bob [5]:**  So if the distance between the points is 7√2, that means they are connected by face diagonals. Therefore, the edge length of the cube is 7. Then the surface area is 6 * 7^2 = 294. 

**Alice [4]:**  So if the edge length is 7, then the surface area of the cube is 6 * (edge length)^2 = 6 * 49 = 294. So the surface area is 294. 



### Work in progress (others)

**Alice [5]:**  I think that's the<...>

**Bob [6]:**  Wait, but I need to<...>



Teacher answer: 294.0 Published blocks: 9
Model-generated role headers: {'Alice': [], 'Bob': []}


## Один тренировочный пример
Ниже полный промпт и все целевые токены по выходным блокам. Это не плоская causal-последовательность:
`trace` дополнительно задаёт, какие блоки и позиции видел каждый токен. Первые 8 строк показаны явно.


In [4]:
tokenizer = AutoTokenizer.from_pretrained(STUDENT)
trace, targets = example["trace"], torch.tensor(example["targets"])
print("PROMPT\n" + tokenizer.decode(teacher["prompt_ids"]))
print("\nTRAINING TARGETS (grouped by output block; prefill/flush are masked)")
for block in sorted({token.block for token in trace.tokens}):
    ids = [
        target
        for token, target in zip(trace.tokens, example["targets"])
        if token.block == block and target != -100
    ]
    if ids:
        print(f"\nBlock {block}, {len(ids)} target tokens:\n{tokenizer.decode(ids)}")
print("\nFirst 8 supervised rows: input → target; position; visible (block, length)")
rows = [
    (token, target) for token, target in zip(trace.tokens, example["targets"]) if target != -100
]
for token, target in rows[:8]:
    print(
        repr(tokenizer.decode([token.token_id])),
        "→",
        repr(tokenizer.decode([target])),
        "position=",
        token.position,
        "view=",
        token.view,
    )
print("Total supervised tokens:", len(rows))

PROMPT
<|im_start|>user
# Collaborative Reasoning

You will collaborate on this problem with another assistant. You will write your thoughts simultaneously with them and collaborate without redundant work. You can collaborate by doing different parts of the problem, double-checking each other's results, trying different approaches, or any other means.

There are 2 assistants, including yourself. You will refer to each other as Alice and Bob.

You will solve the problem together, writing your thoughts in parallel. You will be able to see each other's past and current thoughts as we write them. You will see each other's previous steps as **AssistantName [step]:** <...> .

In the '### Past steps' section, the automated system will gather the thoughts of Alice and Bob as you write them.

After the '### Work in progress (others)' section, you will see the other assistants' unfinished steps. They will write those steps concurrently with you. You will take into account what they are doing. If

## Обучение
Master-веса и моменты AdamW — **BF16 на CPU**, forward/backward — BF16 на GPU.
RoPE buffers остаются FP32. `Replay` восстанавливает видимость и позиции из trace;
`chunk_size=64` ограничивает временную память, не контекст. Loss — средняя CE по немаскированным целям.
`evaluate` из `minisgl.training.sft` проверяет фиксированную teacher-forced траекторию.


In [5]:
torch.manual_seed(7)
replay = Replay.from_trace(
    trace, query_chunk_size=64, attention_checkpointing=True, cache_plans=True
)
student = AutoModelForCausalLM.from_pretrained(STUDENT, dtype=torch.bfloat16).cuda()
student.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
# BF16 master weights on CPU; the GPU model is the compute copy.
master = {
    name: torch.nn.Parameter(p.detach().cpu().clone()) for name, p in student.named_parameters()
}
optimizer = torch.optim.AdamW(master.values(), lr=3e-5, foreach=False)


def evaluate_student():
    metrics = evaluate(student.eval(), [(replay, targets)], chunk_size=64)
    return {key: metrics[key] for key in ("loss", "accuracy")}


print(
    "Master:",
    next(iter(master.values())).dtype,
    "RoPE frequencies:",
    student.model.rotary_emb.inv_freq.dtype,
)
initial = evaluate_student()
print("Initial:", initial)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights:  27%|██▋       | 83/311 [00:00<00:00, 823.13it/s]

Loading weights:  53%|█████▎    | 166/311 [00:00<00:00, 773.46it/s]

Loading weights:  78%|███████▊  | 244/311 [00:00<00:00, 745.31it/s]

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 940.57it/s]

Master: torch.bfloat16 RoPE frequencies: torch.float32


Initial: {'loss': 0.346394419670105, 'accuracy': 0.8805309734513275}


In [6]:
for step in range(1, 81):
    student.train()
    student.zero_grad(set_to_none=True)
    optimizer.zero_grad(set_to_none=True)
    loss = supervised_loss(student, replay, targets, chunk_size=64) / targets.ne(-100).sum()
    loss.backward()
    for name, parameter in student.named_parameters():
        master[name].grad = parameter.grad.detach().cpu()  # BF16, like the master weights.
        parameter.grad = None
    torch.nn.utils.clip_grad_norm_(master.values(), 1.0, error_if_nonfinite=True)
    optimizer.step()
    with torch.no_grad():
        for name, parameter in student.named_parameters():
            parameter.copy_(master[name])
    print(f"Step {step}: loss={loss.item():.6f}")
    if step % 5 == 0:
        final = evaluate_student()
        print("Evaluation:", final)
        if final["accuracy"] == 1.0 and final["loss"] < 0.01:
            break
print("Initial:", initial, "Final:", final)
print("Adam moments:", next(iter(optimizer.state.values()))["exp_avg"].dtype)


Step 1: loss=0.346394


Step 2: loss=0.195146


Step 3: loss=0.069974


Step 4: loss=0.018682


Step 5: loss=0.001509


Evaluation: {'loss': 0.0019253373611718416, 'accuracy': 0.9988938053097345}


... (2 outputs left out)

Step 8: loss=0.013336


Step 9: loss=0.002670


Step 10: loss=0.002032


Evaluation: {'loss': 0.00014722101332154125, 'accuracy': 1.0}
Initial: {'loss': 0.346394419670105, 'accuracy': 0.8805309734513275} Final: {'loss': 0.00014722101332154125, 'accuracy': 1.0}
Adam moments: torch.bfloat16


## Свободная генерация
Student получает только исходный промпт и тот же бюджет 450 раундов. Токены и события учителя не подставляются.
Печатаем его рассуждения, а не только ответ. Совпадение траектории подтверждает запоминание этого примера,
но не обобщение навыка кооперации на другие задачи.
Постоянные checkpoint, логи и каталоги запусков не создаются; временные файлы удаляются автоматически.


In [7]:
# AsyncLLM loads HF files, so use one temporary checkpoint only for this check.
student.cpu()
gc.collect()
torch.cuda.empty_cache()
with TemporaryDirectory() as checkpoint:
    student.save_pretrained(checkpoint)
    tokenizer.save_pretrained(checkpoint)
    result = isolated(solve_parallel, checkpoint, prompt_ids=teacher["prompt_ids"])
display(Markdown(result["transcript"]))
print("Student answer:", result["answer"])
print(
    "Same tokens:",
    all(result[key] == teacher[key] for key in ("history_ids", "current_ids", "answer_ids")),
)
print("Same publications:", result["merge_events"] == teacher["merge_events"])
display(Markdown(result["answer_text"]))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.62s/it]

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.62s/it]

### Past steps

**Bob [1]:** Hi, I'm Bob. I need to find the surface area of the cube given three of its vertices. Let me start by calculating the distances between the points P, Q, and R to see if they form edges, face diagonals, or space diagonals of the cube. 

**Alice [1]:** Hi, I'm Alice. Here's how we can collaborate. I'll start by calculating the distances between the given points to find the edge length of the cube. Let me calculate the distance between P and Q first. The distance formula in 3D is sqrt[(x2-x1)^2 + (y2-y1)^2 + (z2-z1)^2]. So for P=(7,12,10) and Q=(8,8,1), the distance is sqrt[(8-7)^2 + (8-12)^2 + (1-10)^2] = sqrt[1 + 16 + 81] = sqrt[98]. That's approximately 9.899. 

**Bob [2]:**  I'll calculate the distance between Q and R next. Then I can compare the distances to determine if they are edges, face diagonals, or space diagonals. Let me do that. The distance between Q=(8,8,1) and R=(11,3,9) is sqrt[(11-8)^2 + (3-8)^2 + (9-1)^2] = sqrt[9 + 25 + 64] = sqrt[98]. That's the same as the distance between P and Q. So both PQ and QR are sqrt(98). 

**Alice [2]:**  I see. Now I'll calculate the distance between P and R. P=(7,12,10) and R=(11,3,9). The distance is sqrt[(11-7)^2 + (3-12)^2 + (9-10)^2] = sqrt[16 + 81 + 1] = sqrt[98]. So all three distances are sqrt(98). That's interesting. So the three points are all the same distance apart. 

**Bob [3]:**  The distance between P and R is sqrt[(11-7)^2 + (3-12)^2 + (9-10)^2] = sqrt[16 + 81 + 1] = sqrt[98]. So all three distances are the same. That suggests that the three points form an equilateral triangle with side length sqrt(98). But in a cube, three vertices can form an equilateral triangle if they are connected by face diagonals or space diagonals. 

**Bob [4]:**  Let me think. If the three points are connected by edges, then the distance would be 'a'. If they are connected by face diagonals, the distance would be a√2. If they are connected by space diagonals, the distance would be a√3. Since sqrt(98) is approximately 9.899, and 7√2 is about 9.899, that suggests that the distance is a face diagonal. So the edge length of the cube would be 7. 

**Alice [3]:**  So if all three distances are equal, that means the three points are either all connected by edges, face diagonals, or space diagonals. But in a cube, the edges are of length 'a', face diagonals are a√2, and space diagonals are a√3. Since sqrt(98) is approximately 9.899, which is between a√2 and a√3 for a cube with a=7, since 7√2 ≈ 9.899 and 7√3 ≈ 12.124. So sqrt(98) is exactly 7√2. Therefore, the edge length of the cube is 7. 

**Bob [5]:**  So if the distance between the points is 7√2, that means they are connected by face diagonals. Therefore, the edge length of the cube is 7. Then the surface area is 6 * 7^2 = 294. 

**Alice [4]:**  So if the edge length is 7, then the surface area of the cube is 6 * (edge length)^2 = 6 * 49 = 294. So the surface area is 294. 



### Work in progress (others)

**Alice [5]:**  I think that's the<...>

**Bob [6]:**  Wait, but I need to<...>



Student answer: 294.0
Same tokens: True
Same publications: True




Wait, given the limited time, I have to give an answer right now. Considering all my previous attempts, I have to conclude that the final answer is \boxed{294}